In [2]:
# EJERCICIO 1: Deducción Matricial Formal

print("--- EJERCICIO 1: Deducción Matricial ---")

# 1. Definimos las variables simbólicas
theta_i, d_i, a_i, alpha_i = sp.symbols('theta_i d_i a_i alpha_i')

# 2. Definimos las cuatro matrices básicas D-H
Rot_z = sp.Matrix([
    [sp.cos(theta_i), -sp.sin(theta_i), 0, 0],
    [sp.sin(theta_i),  sp.cos(theta_i), 0, 0],
    [0,                0,               1, 0],
    [0,                0,               0, 1]
])

Trans_z = sp.Matrix([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, d_i],
    [0, 0, 0, 1]
])

Trans_x = sp.Matrix([
    [1, 0, 0, a_i],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])

Rot_x = sp.Matrix([
    [1, 0,               0,                0],
    [0, sp.cos(alpha_i), -sp.sin(alpha_i), 0],
    [0, sp.sin(alpha_i),  sp.cos(alpha_i), 0],
    [0, 0,               0,                1]
])

# 3. Multiplicamos las matrices (Rot(z) * Trans(z) * Trans(x) * Rot(x))
# Como en el paso manual, podemos hacerlo todo junto:
T_i = Rot_z * Trans_z * Trans_x * Rot_x

# 4. Demostramos los elementos específicos
# NOTA: Python usa índices desde el 0. Fila 1 es el índice 0, Columna 2 es índice 1.
elemento_f1_c2 = T_i[0, 1]
elemento_f3_c4 = T_i[2, 3]

print(f"Elemento Fila 1, Columna 2: {elemento_f1_c2}")
print(f"Elemento Fila 3, Columna 4: {elemento_f3_c4}\n")

# EJERCICIO 2: Manipulador Planar 3R

print("--- EJERCICIO 2: Cinemática Directa 3R ---")

# 1. Definimos las variables articulares (q) y longitudes de eslabones (l)
q1, q2, q3 = sp.symbols('q1 q2 q3')
l1, l2, l3 = sp.symbols('l1 l2 l3')

# 2. Función para generar la matriz D-H automáticamente evaluando variables
def matriz_dh(theta, d, a, alpha):
    # Evaluamos la matriz general T_i con los valores específicos
    matriz = T_i.subs({theta_i: theta, d_i: d, a_i: a, alpha_i: alpha})
    return matriz

# 3. Construimos las matrices individuales según la tabla D-H
T_01 = matriz_dh(theta=q1, d=0, a=l1, alpha=0)
T_12 = matriz_dh(theta=q2, d=0, a=l2, alpha=0)
T_23 = matriz_dh(theta=q3, d=0, a=l3, alpha=0)

# 4. Calculamos la cinemática directa total (Multiplicación de T01 * T12 * T23)
T_03 = T_01 * T_12 * T_23

# 5. Simplificamos trigonométricamente
# sp.trigsimp agrupa las sumas de senos y cosenos (ej: cos(q1)cos(q2) - sin(q1)sin(q2) = cos(q1+q2))
T_03_simplificada = sp.trigsimp(T_03)

print("Matriz de Cinemática Directa (T_03) simplificada:")
sp.pprint(T_03_simplificada) # pprint imprime la matriz en un formato legible (pretty print)

--- EJERCICIO 1: Deducción Matricial ---
Elemento Fila 1, Columna 2: -sin(theta_i)*cos(alpha_i)
Elemento Fila 3, Columna 4: d_i

--- EJERCICIO 2: Cinemática Directa 3R ---
Matriz de Cinemática Directa (T_03) simplificada:
⎡cos(q₁ + q₂ + q₃)  -sin(q₁ + q₂ + q₃)  0  l₁⋅cos(q₁) + l₂⋅cos(q₁ + q₂) + l₃⋅c ↪
⎢                                                                              ↪
⎢sin(q₁ + q₂ + q₃)  cos(q₁ + q₂ + q₃)   0  l₁⋅sin(q₁) + l₂⋅sin(q₁ + q₂) + l₃⋅s ↪
⎢                                                                              ↪
⎢        0                  0           1                           0          ↪
⎢                                                                              ↪
⎣        0                  0           0                           1          ↪

↪ os(q₁ + q₂ + q₃)⎤
↪                 ⎥
↪ in(q₁ + q₂ + q₃)⎥
↪                 ⎥
↪                 ⎥
↪                 ⎥
↪                 ⎦


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
import ipywidgets as widgets

# Parámetros del Robot

L1 = 2.0
L2 = 1.5
L3 = 1.0


# Función de Cinemática Directa

def calcular_posiciones(q1, q2, q3):
    x0, y0 = 0, 0
    x1 = L1 * np.cos(q1)
    y1 = L1 * np.sin(q1)
    x2 = x1 + L2 * np.cos(q1 + q2)
    y2 = y1 + L2 * np.sin(q1 + q2)
    x3 = x2 + L3 * np.cos(q1 + q2 + q3)
    y3 = y2 + L3 * np.sin(q1 + q2 + q3)
    return [x0, x1, x2, x3], [y0, y1, y2, y3]

# Función para Dibujar (se llamará con cada cambio)

def dibujar_robot(q1_deg, q2_deg, q3_deg):
    # Convertimos los grados del slider a radianes
    q1 = np.deg2rad(q1_deg)
    q2 = np.deg2rad(q2_deg)
    q3 = np.deg2rad(q3_deg)
    
    xs, ys = calcular_posiciones(q1, q2, q3)
    
    # Configuramos el gráfico
    plt.figure(figsize=(7, 7))
    limite = L1 + L2 + L3 + 0.5
    plt.xlim(-limite, limite)
    plt.ylim(-limite, limite)
    
    # Dibujamos los elementos
    plt.plot(xs, ys, 'o-', color='#004b87', linewidth=4, markersize=10, markerfacecolor='white', markeredgewidth=2)
    plt.plot(xs[0], ys[0], 'ks', markersize=12) # Base
    plt.plot(xs[-1], ys[-1], 'ro', markersize=10) # TCP
    
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.title("Simulación Planar 3R (Google Colab)")
    plt.gca().set_aspect('equal')
    plt.show()

# Generar los Sliders Interactivos

interact(dibujar_robot, 
         q1_deg=widgets.IntSlider(min=-90, max=180, step=1, value=0, description='q1 (Base)'),
         q2_deg=widgets.IntSlider(min=-180, max=180, step=1, value=0, description='q2 (Codo)'),
         q3_deg=widgets.IntSlider(min=-180, max=180, step=1, value=0, description='q3 (Muñeca)'))

interactive(children=(IntSlider(value=0, description='q1 (Base)', max=180, min=-90), IntSlider(value=0, descri…

<function __main__.dibujar_robot(q1_deg, q2_deg, q3_deg)>

In [2]:
'''
Ejercicio 3 y 4: Implementación Simbólica, Numérica y Validación
'''
import numpy as np
import sympy as sp

# 1. Módulo de Funciones D-H (dh_builder.py)

def dh_matrix_symbolic(theta, d, a, alpha):
    
# Construye analíticamente la matriz D-H estándar (4x4) usando SymPy.
    
    return sp.Matrix([
        [sp.cos(theta), -sp.sin(theta)*sp.cos(alpha),  sp.sin(theta)*sp.sin(alpha), a*sp.cos(theta)],
        [sp.sin(theta),  sp.cos(theta)*sp.cos(alpha), -sp.cos(theta)*sp.sin(alpha), a*sp.sin(theta)],
        [0,              sp.sin(alpha),               sp.cos(alpha),              d],
        [0,              0,                           0,                          1]
    ])

def dh_matrix_numeric(theta_deg, d, a, alpha_deg):
    
#Evalúa numéricamente la matriz D-H estándar (4x4) usando NumPy.
#Los ángulos de entrada se reciben en grados.
    
    th = np.radians(theta_deg)
    al = np.radians(alpha_deg)
    
    return np.array([
        [np.cos(th), -np.sin(th)*np.cos(al),  np.sin(th)*np.sin(al), a*np.cos(th)],
        [np.sin(th),  np.cos(th)*np.cos(al), -np.cos(th)*np.sin(al), a*np.sin(th)],
        [0.0,         np.sin(al),            np.cos(al),           d],
        [0.0,         0.0,                   0.0,                  1.0]
    ], dtype=np.float64)

def fk_chain(dh_table):
    
# Calcula la cinemática directa total encadenando una lista de tuplas/filas D-H:
# dh_table: lista de tuplas [(theta_1, d_1, a_1, alpha_1), ...]
    
    T_total = np.eye(4, dtype=np.float64)
    for row in dh_table:
        theta, d, a, alpha = row
        T_i = dh_matrix_numeric(theta, d, a, alpha)
        T_total = T_total @ T_i
    return T_total


# 2. Validación Numérica del Caso de Prueba

# Parámetros asignados:
theta_1 = 45.0    # grados
d_1 = 0.35        # metros
a_1 = 0.15        # metros
alpha_1 = -90.0   # grados

# Evaluación de la matriz homogénea 4x4
T_1 = dh_matrix_numeric(theta_1, d_1, a_1, alpha_1)

# Extracción de la submatriz de rotación R (3x3) y del vector de posición p (3x1)
R = T_1[:3, :3]
p_calculado = T_1[:3, 3]

# Cálculo de las métricas de validación
I3 = np.eye(3)
norma_frobenius = np.linalg.norm(R.T @ R - I3, ord='fro')
det_R = np.linalg.det(R)
error_det = abs(det_R - 1.0)

# Vector de posición teórico esperado p = [a1*cos(theta1), a1*sin(theta1), d1]^T
th_rad = np.radians(theta_1)
p_teorico = np.array([a_1 * np.cos(th_rad), a_1 * np.sin(th_rad), d_1])
error_p = np.linalg.norm(p_calculado - p_teorico)

print("=" * 65)
print("MATRIZ HOMOGÉNEA EVALUADA T_1:")
print("=" * 65)
print(np.round(T_1, 6))
print()

print("=" * 65)
print("VALIDACIÓN DE ORTOGONALIDAD Y POSICIÓN:")
print("=" * 65)
print(f"1. ||R^T * R - I||_F:     {norma_frobenius:.2e}  ->  ¿< 1e-15?: {norma_frobenius < 1e-15}")
print(f"2. |det(R) - 1.0|:         {error_det:.2e}  ->  ¿< 1e-15?: {error_det < 1e-15}")
print(f"3. Vector p calculado:     {np.round(p_calculado, 6)}")
print(f"   Vector p teórico:       {np.round(p_teorico, 6)}")
print(f"   Diferencia ||p - p_th||: {error_p:.2e}  ->  ¿Coinciden?: {np.isclose(error_p, 0.0)}")
print("=" * 65)

MATRIZ HOMOGÉNEA EVALUADA T_1:
[[ 0.707107 -0.       -0.707107  0.106066]
 [ 0.707107  0.        0.707107  0.106066]
 [ 0.       -1.        0.        0.35    ]
 [ 0.        0.        0.        1.      ]]

VALIDACIÓN DE ORTOGONALIDAD Y POSICIÓN:
1. ||R^T * R - I||_F:     3.20e-16  ->  ¿< 1e-15?: True
2. |det(R) - 1.0|:         0.00e+00  ->  ¿< 1e-15?: True
3. Vector p calculado:     [0.106066 0.106066 0.35    ]
   Vector p teórico:       [0.106066 0.106066 0.35    ]
   Diferencia ||p - p_th||: 0.00e+00  ->  ¿Coinciden?: True
